In [1]:
!pip install pandas sentence-transformers Levenshtein jellyfish spaCy scikit-learn jiwer joblib
!python -m spacy download es_core_news_md

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 360.5/360.5 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 50.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 MB 14.3 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import pandas as pd
import re
import string
import spacy
import jellyfish
import Levenshtein
import jiwer
import joblib
from sentence_transformers import SentenceTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report

print("Loading NLP Models...")
model = SentenceTransformer('intfloat/multilingual-e5-base')
nlp = spacy.load("es_core_news_md")

# --- 1. NLP & CLEANING FUNCTIONS ---

def clean_transcript(text):
    if pd.isna(text): return ""
    text = str(text)

    # Remove ALL transcriber notes in brackets and parentheses
    text = re.sub(r'\[.*?\]', ' ', text)
    text = re.sub(r'\(.*?\)', ' ', text)

    # Replace hesitation pauses or hyphens with a SPACE
    text = re.sub(r'\.+', ' ', text)
    text = re.sub(r'-+', ' ', text)

    # Remove "xxx" denoting unintelligible audio
    text = re.sub(r'\bx+\b', ' ', text, flags=re.IGNORECASE)

    # Remove English and Spanish punctuation
    spanish_punct = string.punctuation + '¿¡'
    text = text.translate(str.maketrans('', '', spanish_punct))

    # Flatten vowels but PRESERVE the ñ
    replacements = {
        'á': 'a', 'é': 'e', 'í': 'i', 'ó': 'o', 'ú': 'u', 'ü': 'u',
        'Á': 'A', 'É': 'E', 'Í': 'I', 'Ó': 'O', 'Ú': 'U', 'Ü': 'U'
    }
    for accented, flat in replacements.items():
        text = text.replace(accented, flat)

    return " ".join(text.split()).lower()

def check_critical_meaning_loss(target, utterance):
    t_doc = nlp(target)
    u_doc = nlp(utterance)

    t_neg = any(token.text == "no" for token in t_doc)
    u_neg = any(token.text == "no" for token in u_doc)
    if t_neg != u_neg: return True

    antonyms = [{"derecha", "izquierda"}, {"arriba", "abajo"}, {"antes", "despues"}]
    for pair in antonyms:
        word1, word2 = list(pair)
        if (word1 in target and word2 in utterance) or (word2 in target and word1 in utterance):
            return True
    return False

def get_token_phonetic_similarity(target, utterance):
    t_words = target.split()
    u_words = utterance.split()
    if not t_words: return 0.0
    total_score = sum(max([jellyfish.jaro_winkler_similarity(tw, uw) for uw in u_words] + [0.0]) for tw in t_words)
    return total_score / len(t_words)

def get_lexical_similarity(target, utterance):
    if not target or not utterance: return 0.0
    return Levenshtein.ratio(target, utterance)

def get_semantic_similarity(target, utterance):
    if not target or not utterance: return 0.0
    embeddings = model.encode([target, utterance])
    return model.similarity(embeddings[0], embeddings[1]).item()

def get_wer_features(target, utterance):
    """
    Returns 4 distinct metrics instead of a basic 'length penalty'.
    This tells the AI if the student added extra words (Insertions),
    missed words (Deletions), or swapped words (Substitutions).
    """
    if not target or not utterance:
        return 1.0, 0.0, 1.0, 0.0
    try:
        # Calculate WER alignment
        out = jiwer.process_words(target, utterance)

        # Normalize the metrics based on the target length
        t_len = len(target.split())
        if t_len == 0: t_len = 1

        wer = out.wer
        insertions = out.insertions / t_len
        deletions = out.deletions / t_len
        substitutions = out.substitutions / t_len

        return wer, insertions, deletions, substitutions
    except Exception as e:
        print(f"WER Error on Target: '{target}' | Utterance: '{utterance}' | Error: {e}")
        return 1.0, 0.0, 1.0, 0.0

# --- 2. TRAIN THE AI ON HISTORICAL DATA ---

print("\nAggregating historical training data...")
train_file_path = 'Example_EIT Transcription and Scoring Sheet.xlsx'
all_train_sheets = pd.read_excel(train_file_path, sheet_name=None)

data_frames = []
for sheet_name, df in all_train_sheets.items():
    if '_vA' in sheet_name or '_vB' in sheet_name:
        try:
            temp_df = df[['Stimulus', 'Transcription Rater 1', 'Score Rater 1']].copy()
            temp_df.columns = ['Stimulus', 'Transcription', 'Human_Score']
            data_frames.append(temp_df)
        except KeyError:
            continue

train_df = pd.concat(data_frames, ignore_index=True)
train_df = train_df.dropna(subset=['Human_Score', 'Stimulus', 'Transcription'])
train_df['Human_Score'] = pd.to_numeric(train_df['Human_Score'], errors='coerce')
train_df = train_df.dropna(subset=['Human_Score']).astype({'Human_Score': 'int'})

print("Extracting NLP Features for AI Training...")
train_df['Cleaned_Target'] = train_df['Stimulus'].apply(clean_transcript)
train_df['Cleaned_Utterance'] = train_df['Transcription'].apply(clean_transcript)

train_df['Lexical'] = train_df.apply(lambda row: get_lexical_similarity(row['Cleaned_Target'], row['Cleaned_Utterance']), axis=1)
train_df['Semantic'] = train_df.apply(lambda row: get_semantic_similarity(row['Cleaned_Target'], row['Cleaned_Utterance']), axis=1)
train_df['Phonetic'] = train_df.apply(lambda row: get_token_phonetic_similarity(row['Cleaned_Target'], row['Cleaned_Utterance']), axis=1)
train_df['Antonym_Swap'] = train_df.apply(lambda row: int(check_critical_meaning_loss(row['Cleaned_Target'], row['Cleaned_Utterance'])), axis=1)
train_df[['WER', 'Insertions', 'Deletions', 'Substitutions']] = train_df.apply(
    lambda row: pd.Series(get_wer_features(row['Cleaned_Target'], row['Cleaned_Utterance'])), axis=1
)
print("Training Random Forest Classifier...")

X = train_df[['Lexical', 'Semantic', 'Phonetic', 'Antonym_Swap', 'WER', 'Insertions', 'Deletions', 'Substitutions']]
y = train_df['Human_Score']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define the grid of hyperparameters to test
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [4, 6, 8, 10],
    'class_weight': ['balanced']
}

# Run 5-Fold Cross Validation GridSearch
grid_search = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)

# Automatically extract the best performing model
best_rf_model = grid_search.best_estimator_

print(f"Best Settings Found: {grid_search.best_params_}")

# Using the optimal parameters found with GridSearchCV!
print("ML Engine Trained Successfully!")

# Test the best model
y_pred = best_rf_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"AI Agreement with Human Raters (Cross-Validated Accuracy): {accuracy * 100:.2f}%\n")
print(classification_report(y_test, y_pred, zero_division=0))

# --- 3. SCORE THE TEST FILES ---

print("\nScoring the final Evaluation Test Participants...")
test_file_path = 'AutoEIT Sample Transcriptions for Scoring.xlsx'
all_test_sheets = pd.read_excel(test_file_path, sheet_name=None)

# Export the final predictions to a new Excel file
with pd.ExcelWriter('AutoEIT_Developer_Matrix_Full_Data.xlsx') as writer:
    for sheet_name, test_df in all_test_sheets.items():
        if sheet_name == 'Info':
            continue

        print(f" -> Grading Participant: {sheet_name}")

        # Clean Test Data
        test_df['Cleaned_Target'] = test_df['Stimulus'].apply(clean_transcript)
        test_df['Cleaned_Utterance'] = test_df['Transcription Rater 1'].apply(clean_transcript)

        # Extract features for Test Data
        test_df['Lexical'] = test_df.apply(lambda row: get_lexical_similarity(row['Cleaned_Target'], row['Cleaned_Utterance']), axis=1)
        test_df['Semantic'] = test_df.apply(lambda row: get_semantic_similarity(row['Cleaned_Target'], row['Cleaned_Utterance']), axis=1)
        test_df['Phonetic'] = test_df.apply(lambda row: get_token_phonetic_similarity(row['Cleaned_Target'], row['Cleaned_Utterance']), axis=1)
        test_df['Antonym_Swap'] = test_df.apply(lambda row: int(check_critical_meaning_loss(row['Cleaned_Target'], row['Cleaned_Utterance'])), axis=1)
        test_df[['WER', 'Insertions', 'Deletions', 'Substitutions']] = test_df.apply(
    lambda row: pd.Series(get_wer_features(row['Cleaned_Target'], row['Cleaned_Utterance'])), axis=1
)

        # Predict the scores using our AI
        X_test = test_df[['Lexical', 'Semantic', 'Phonetic', 'Antonym_Swap', 'WER', 'Insertions', 'Deletions', 'Substitutions']]
        test_df['Predicted_Score'] = best_rf_model.predict(X_test)

        # Save to the new Excel file
        test_df.to_excel(writer, sheet_name=sheet_name, index=False)

print("\n All done! 'AutoEIT_Developer_Matrix_Full_Data.xlsx' ")
joblib.dump(best_rf_model, 'autoeit_model.pkl')

Loading NLP Models...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]


Aggregating historical training data...
Extracting NLP Features for AI Training...
Training Random Forest Classifier...
Best Settings Found: {'class_weight': 'balanced', 'max_depth': 8, 'n_estimators': 300}
ML Engine Trained Successfully!
AI Agreement with Human Raters (Cross-Validated Accuracy): 90.38%

              precision    recall  f1-score   support

           0       0.94      0.94      0.94        17
           1       0.62      0.73      0.67        11
           2       0.59      0.52      0.55        25
           3       0.81      0.88      0.84        64
           4       0.99      0.97      0.98       195

    accuracy                           0.90       312
   macro avg       0.79      0.81      0.80       312
weighted avg       0.91      0.90      0.90       312


Scoring the final Evaluation Test Participants...
 -> Grading Participant: 38001-1A
 -> Grading Participant: 38002-2A
 -> Grading Participant: 38004-2A
 -> Grading Participant: 38006-2A

 All done! 'Auto

['autoeit_model.pkl']